# Baseline Models for Daily Rat Sightings in Manhattan

In [14]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import datetime

from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error

from pandas.tseries.holiday import USFederalHolidayCalendar

import warnings
from statsmodels.tools.sm_exceptions import ConvergenceWarning
warnings.simplefilter('ignore', ConvergenceWarning)


# Modeling Daily Rat Sightings in Manhattan

## Importing the Data

In [15]:
# set up the time series split
tscv = TimeSeriesSplit(gap=0, max_train_size=None, n_splits=26, test_size=14)

rs = pd.read_csv('../../scr/data/cleaned_rat_sightings_data/all_daily_borough_rs.csv')
rs['created_date'] = pd.to_datetime(rs['created_date']) 

# Start by cutting off data before 2020-01-01 and after 2025-02-28.
rs = rs[rs['created_date']<'2025-03-01']
rs = rs[rs['created_date']>='2020-01-01']

## Restrict to MANHATTAN

rs = rs[rs['borough']=='MANHATTAN']

## Drop the column with borough

rs = rs.drop(columns=['borough'])

rs

,created_date,count
2,2020-01-01,4
7,2020-01-02,7
12,2020-01-03,16
17,2020-01-04,10
21,2020-01-05,5
...,...,...
8905,2025-02-24,19
8910,2025-02-25,20
8915,2025-02-26,15
8920,2025-02-27,16


In [16]:
## We find the missing row and add it in.
## There's probably a better way to find in the missing dates and update it. This is a sort of hacky solution.

date_range = pd.date_range(start=rs['created_date'].min(), end=rs['created_date'].max(), freq='D')
complete_dates_df = pd.DataFrame(date_range, columns=['created_date'])
rs = pd.merge(complete_dates_df, rs, on='created_date', how='left')
rs['count'] = rs['count'].fillna(0).astype(int)
rs = rs.sort_values(by='created_date').reset_index(drop=True)

rs

,created_date,count
0,2020-01-01,4
1,2020-01-02,7
2,2020-01-03,16
3,2020-01-04,10
4,2020-01-05,5
...,...,...
1881,2025-02-24,19
1882,2025-02-25,20
1883,2025-02-26,15
1884,2025-02-27,16


## Baseline Seasonal Average Model

In [17]:
years_back_use = 4
day_window_use = 4

In [18]:
def seasonal_average_forecast(data, target_dates, years_back=years_back_use, day_window=day_window_use):
    df = data.copy()
    # ensure datetime type
    df["created_date"] = pd.to_datetime(df["created_date"])
    df["doy"] = df["created_date"].dt.dayofyear
    df["year"] = df["created_date"].dt.year

    forecasts = []
    for target_date in target_dates:
        target_doy = target_date.dayofyear
        target_year = target_date.year
        mask = ((df["year"] >= target_year - years_back) & (df["year"] < target_year) & (np.abs(df["doy"] - target_doy) <= day_window))
        forecasts.append(df.loc[mask, "count"].mean())
    return pd.Series(forecasts, index=target_dates)

In [19]:
results = []

rs["created_date"] = pd.to_datetime(rs["created_date"])

for i, (train_index, test_index) in enumerate(tscv.split(rs)):
    
    train = rs.iloc[train_index].copy()
    test = rs.iloc[test_index].copy()
    
    # Target dates = the dates we want to forecast. There are 14 days.
    target_dates = test["created_date"]
    
    # Seasonal forecast using only the training data (we will go back 5 years and take the average and use a day_window of 5 as well.)
    y_pred = seasonal_average_forecast(data=train, target_dates=target_dates, years_back=years_back_use,day_window=day_window_use)
    y_pred = np.round(y_pred)
    # We take the true values.
    y_true = test["count"].values
    
    # Compute the metrics
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = mean_absolute_percentage_error(y_true, y_pred)
    
    # Append the results of the metrics to the table as well as the fold number.
    results.append({"fold": i, "rmse": rmse, "mape": mape})

# Convert the data to a table for readability.
baseline_results_df = pd.DataFrame(results)

# We also include a new row which consists of the average RMSE and MAPE over each fold.
baseline_results_df.loc["mean"] = ["mean", baseline_results_df["rmse"].mean(), baseline_results_df["mape"].mean()]

baseline_results_df

,fold,rmse,mape
0,0,3.835920,0.265159
1,1,3.900549,0.236755
2,2,5.561346,0.551697
3,3,5.257647,0.317101
4,4,5.271216,0.518104
5,5,6.606274,0.393644
6,6,6.938505,0.371349
7,7,5.644213,0.338521
8,8,6.475228,0.490078
9,9,5.457629,0.273051


## Year Ago Rolling 4 Week Average 

In [20]:
rs.rename(columns={'created_date': 'ds', 'count': 'y'}, inplace=True)

## Just saving a copy for later
rs_saved = rs.copy()

In [21]:
# Tired of writing np.sqrt or typing a long name.
def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

results = []

for fold, (train_index, test_index) in enumerate(tscv.split(rs)):
    train = rs.iloc[train_index]
    test = rs.iloc[test_index]

    # Calculate the 4-week rolling average for the training data
    train_sorted = train.sort_values('ds') # making sure to sort it by date
    train_sorted['rolling_4w'] = train_sorted['y'].rolling(window=4, min_periods=1).mean()

    # This part of the code makes the predictions. We use the 'rolling_4w' column of the training set.
    y_pred = []
    y_true = test['y'].values

    for idx, row in test.iterrows():
        # Predict using the latest rolling average from the train data
        prediction = train_sorted['rolling_4w'].iloc[-1]  # Last value in the train rolling avg
        y_pred.append(prediction)
        
    # Calculate RMSE and MAPE for this fold
    fold_rmse = rmse(y_true, np.round(y_pred))
    fold_mape = mean_absolute_percentage_error(y_true, np.round(y_pred))
    
    results.append({'fold': fold, 'rmse': fold_rmse, 'mape': fold_mape})

rolling4w_results_df = pd.DataFrame(results)

# Optional: add a row for the overall average RMSE and MAPE
overall_rmse = rolling4w_results_df['rmse'].mean()
overall_mape = rolling4w_results_df['mape'].mean()
rolling4w_results_df.loc['mean'] = ['mean', overall_rmse, overall_mape]

In [22]:
rolling4w_results_df

,fold,rmse,mape
0,0,3.615443,0.240564
1,1,4.097037,0.272435
2,2,5.358571,0.534791
3,3,7.473764,0.315292
4,4,6.313251,0.552352
5,5,7.131419,0.452797
6,6,7.657862,0.447305
7,7,5.264436,0.300166
8,8,5.732115,0.420732
9,9,6.141196,0.283705


## Results of the Two Baselines

In [23]:
models = {
    'baseline': baseline_results_df,
    'rolling4w': rolling4w_results_df,
}

all_results = []
for model_name, df in models.items():
    df['model'] = model_name
    all_results.append(df)

# Put all of the dataframes together into one dataframe for display
final_results_df = pd.concat(all_results, ignore_index=True)
# Make a pivot table so that we display rmse, mape and then each of the models and their results.
final_table = final_results_df.pivot(index='fold', columns='model', values=['rmse', 'mape'])
final_table.index = final_table.index.where(final_table.index != '-', 'mean')

final_table

rmse                mape          
model  baseline rolling4w  baseline rolling4w
fold                                         
0      3.835920  3.615443  0.265159  0.240564
1      3.900549  4.097037  0.236755  0.272435
2      5.561346  5.358571  0.551697  0.534791
3      5.257647  7.473764  0.317101  0.315292
4      5.271216  6.313251  0.518104  0.552352
5      6.606274  7.131419  0.393644  0.452797
6      6.938505  7.657862  0.371349  0.447305
7      5.644213  5.264436  0.338521  0.300166
8      6.475228  5.732115  0.490078  0.420732
9      5.457629  6.141196  0.273051  0.283705
10     5.732115  5.311712  0.196257  0.195805
11     6.141196  5.203021  0.344191  0.278515
12     5.424811  5.291503  0.247036  0.235474
13     4.326001  5.021383  0.217624  0.265180
14     6.436503  6.117889  0.379745  0.321291
15     6.324555  6.933356  0.333129  0.397371
16     5.358571  5.567764  0.339680  0.373150
17     5.756983  6.486249  0.427821  0.502809
18     3.484660  2.329929  0.224797  0.136535
19     4.551295  5.444525  0.470164  0.591072
20     4.407785  4.026697  1.109583  0.917700
21     4.174754  5.548488  0.753362  1.016573
22     4.621379  3.391165  0.385759  0.271950
23     7.396911  7.225945  0.555416  0.531086
24     4.415880  8.639610  0.456439  0.946572
25     5.056820  5.077964  0.553271  0.547603
mean   5.329183  5.630858  0.413451  0.436493